# Module 2 — Deploy the agent to AgentCore Runtime (with observability)

In Module 1 you ran the text-to-SQL agent locally. Now you'll deploy the **same agent** to
**Amazon Bedrock AgentCore Runtime** so it runs as a managed, HTTP-invocable service — and you'll
see its execution **traced in CloudWatch**, with no extra agent code.

Unlike a "here's a pre-built project, just deploy it" walkthrough, this notebook builds the whole
thing **end-to-end with the AgentCore CLI** so you see where every piece comes from:

`agentcore create` (scaffold) → configure → `agentcore deploy` → `agentcore invoke` → `agentcore traces`


## How the deploy reuses Module 1 — no rewrite

The deployed agent's brain is the **same `build_agent_options()`** from Module 1's `agent.py`.
The only new file is a thin entrypoint, `analytics_agent/agent_agentcore.py`:

- it's decorated with `@app.entrypoint` (the AgentCore HTTP contract),
- it calls `build_agent_options()` — the single source of truth — with **zero** agent logic of its own,
- it adds deploy-only plumbing: uploading any files the agent produces to S3 and returning presigned URLs.

A drift-guard test asserts this bundle's `agent.py` is byte-identical to Module 1's.

## Setup

Run the cell below to install the AgentCore CLI (Node), run `npm ci` for the CDK project, sync the
Python env, and register the kernel. After it completes, **select the `agentic-analytics-module-2-deploy`
kernel** from the kernel picker (top-right) and continue with the rest of the notebook.

### Setup step 1

When you run the first script, it will ask you to select a environment

![](images/select-system-python.png)

and you can select the global env for now, and in this case it is 3.11.15 but this version may change

![](images/select-python-global-env.png)

once selected, you can rerun the setup.sh script

In [1]:
!bash setup.sh

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇npm warn deprecated inflight@1.0.6: This module is not supported, and leaks memory. Do not use it. Check out lru-cache if you want a good and tested way to coalesce async requests by a key value, which is much more comprehensive and powerful.
⠇⠏npm warn deprecated glob@7.2.3: Old versions of glob are not supported, and contain widely publicized security vulnerabilities, which have been fixed in the current version. Please update. Support for old versions may be purchased (at exorbitant rates) by contacting i@izs.me
⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 336 packages, and audited 374 packages in 6s
⠹
⠹46 packages are looking for funding
⠹  run `npm fund` for details
⠹
1 moderate severity vulnerability

To address all issues, run:
  npm audit fix

Run `npm audit` for details.
⠹Using CPython 3.11.14
Creating virtual environment at: .venv
Resolved 176 packages in 256ms                                       
Installed 154 packages in 466ms52                    

### Setup step 2

Once you see the dependencies and kernel spec `agentic-analytics-module-2-deploy` are installed per the message from the last step, please refresh your browser (not refresh kernel but browser)

![](images/refresh-browser.png)

and once refreshed, click on the button (it probably shows a python version 3.11.15) you used to select kernel in the preview section

![](images/current-python.png)

it will show you the option to select another kernel and please click

![](images/select-another-kernel.png)

once clicked, you will see the option to select a Jupyter kernel — please click on "Jupyter Kernel"

![](images/select-jupyter-kernel.png)

once clicked, you can see our registered module kernel. The screenshot below shows module-1 as an example, but ***please select `agentic-analytics-module-2-deploy` since you are working on Module 2***

![](images/example-select-module-1-jupter-kernel.png)

once selected, you will see it as your active kernel. Again the screenshot shows module-1 as an example, ***please select accordingly depending on which module you are working on — for this module, select `agentic-analytics-module-2-deploy`***

![](images/example-module-1-jupyter-kernel-selected.png)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

import boto3
acct = boto3.client("sts").get_caller_identity()["Account"]
region = os.getenv("AWS_REGION", "us-west-2")
print("Account:", acct, "| Region:", region)

## Step 1 — Scaffold the AgentCore project (`agentcore create`)

`agentcore create` lays down the project skeleton: `agentcore/agentcore.json` (the resource spec)
and `agentcore/cdk/` (the CDK app that provisions everything). We use `--no-agent` because we'll
attach our **existing** `analytics_agent/` bundle in the next step rather than generating a template.

> **Note:** this repo already ships a known-good `agentcore/` so the module works out of the box.

In [ ]:
import json
print(json.dumps(json.load(open("agentcore/agentcore.json"))["runtimes"][0], indent=2))

{
  "name": "analytics",
  "build": "Container",
  "entrypoint": "agent_agentcore.py",
  "codeLocation": "analytics_agent/",
  "runtimeVersion": "PYTHON_3_11",
  "networkMode": "PUBLIC",
  "protocol": "HTTP",
  "instrumentation": {
    "enableOtel": true
  },
  "envVars": [
    {
      "name": "CLAUDE_CODE_USE_BEDROCK",
      "value": "1"
    },
    {
      "name": "ANTHROPIC_MODEL",
      "value": "global.anthropic.claude-opus-4-6-v1"
    },
    {
      "name": "ANTHROPIC_SMALL_FAST_MODEL",
      "value": "global.anthropic.claude-haiku-4-5-20251001-v1:0"
    },
    {
      "name": "ATHENA_DATABASE",
      "value": "student_analytics"
    }
  ]
}


## Step 2 — Configure: observability + Bedrock + Athena

Two things make this runtime work:

1. **`instrumentation.enableOtel: true`** — this single flag turns on tracing. The CLI's container
   template runs the agent under `opentelemetry-instrument`, and `agentcore deploy` enables
   **CloudWatch Transaction Search** for you. No hand-written spans, no manual console toggle.
2. **`envVars`** — `CLAUDE_CODE_USE_BEDROCK=1`, the Bedrock model id, and `ATHENA_DATABASE`. The
   Athena results bucket is **derived from your account id at runtime**, so nothing account-specific
   is baked into committed config.

The agent also needs Athena/Glue/S3 permissions at runtime. The CDK auto-creates the runtime role
with only Bedrock + Logs; we add the data-plane permissions in `agentcore/cdk/lib/cdk-stack.ts`, so
**every deploy gets them automatically**.

In [ ]:
# Set your deployment target (account + region). aws-targets.json is gitignored.
import json
targets = [{"name": "default", "description": "my target",
            "account": acct, "region": region}]
with open("agentcore/aws-targets.json", "w") as f:
    json.dump(targets, f, indent=2)
print("wrote agentcore/aws-targets.json →", acct, region)
print(json.dumps(targets, indent=2))


## Step 3 — (Optional) Run it locally first: `agentcore dev`

`agentcore dev` runs the container's HTTP contract locally so you can smoke-test before deploying.
It's a long-running server, so run it in a **terminal** under this folder (not a notebook cell):

```bash
agentcore dev
# then in another shell:
curl -s -XPOST http://localhost:8081/invocations \
    -H "Content-Type: application/json" \
    -d '{"prompt":"How many distinct students are enrolled?"}'
```
or 
```bash
agentcore dev "How many distinct students are enrolled?" -p 8081
```

> **Note:** You'll see `ERROR:opentelemetry.exporter.otlp...403 Forbidden` in the dev logs — this
> is safe to ignore. Locally there is no CloudWatch OTEL collector to receive spans, so the exporter
> fails. The agent itself works fine; tracing only delivers spans when deployed to AgentCore (Step 6).

## Step 4 — Deploy (`agentcore deploy`)

This builds the container in the cloud (CodeBuild, ARM64 — no local Docker needed), provisions the
runtime + IAM role via CDK, and enables Transaction Search. Takes a few minutes.

> Your account/region must be CDK-bootstrapped once: `npx cdk bootstrap aws://<account>/<region>`.

In [ ]:
!agentcore deploy -y

✓ Load deployment target
⠋ Validate project...(node:348186) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
(Use `node --trace-warnings ...` to show where the warning was created)
✓ Validate project
✓ Build CDK project...
⠋ Synthesize CloudFormation...(node:348186) Warning: NodeVersionSupportWarning: The AWS SDK for JavaScript (v3)
versions published after the first week of January 2027
will require node >=22. You are running node v20.20.2.

To continue receiving updates to AWS services, bug fixes,
and security updates please upgrade to node >=22.

More information can be found at: https://a.co/c895JFp
✓ Synthesize CloudFormation
✓ Check bootstrap status...
✓ Check stack status...
⠴ Deploy to AW

## Step 5 — Invoke the deployed agent (`agentcore invoke`)

Send a real question. Use a session id of **≥33 characters** (a runtime requirement). The agent
loads a skill, reads table metadata, writes SQL, runs it on Athena, and answers.

In [ ]:
!agentcore invoke '{"prompt": "How many distinct students are enrolled? Give me the number."}' \
    --session-id agentic-analytics-m2-demo-session-0001

In [ ]:
!agentcore status

## Step 6 — See the trace (`agentcore traces`)

Because `enableOtel` was on, the invocation emitted a trace to **CloudWatch GenAI Observability** —
no extra code. List recent traces (and get the console deep-link); spans take ~2-3 minutes to index.

In [ ]:
!agentcore traces list --runtime analytics --since 1h

In [7]:
!echo $AWS_REGION

us-east-1


## View the traces

**In the console (the main event):** open the GenAI Observability dashboard — it has **Agents**,
**Sessions**, and **Traces** views. Pick your agent, drill into a session, and open a trace to see the
span waterfall (tool calls, token usage, latency).

```
https://{AWS_REGION}.console.aws.amazon.com/cloudwatch/home?region={AWS_REGION}#gen-ai-observability/sessions
```

(Replace `{AWS_REGION}`. Allow ~2–10 minutes after invoking for spans to be indexed.)


![](images/trace-span.png)


## Recap

- The **same** `build_agent_options()` runs locally (Module 1) and deployed — the entrypoint is a
  thin `@app.entrypoint` wrapper, no agent logic duplicated.
- **Observability = one flag.** `enableOtel: true` + `agentcore deploy` gives you CloudWatch traces
  with zero hand-written spans, zero manual console steps.
- The runtime's Athena/Glue/S3 permissions are added in the CDK stack, so deploys are reproducible.

**Next:** Module 3 adds a follow-up-questions (clarification) workflow to the same agent.